In [0]:
import numpy as np
import pandas as pd
from sklearn.metrics import precision_score, recall_score
from scipy import stats

from pyspark.sql.functions import lit

In [0]:
INSERT FULL PROFESSORS AND ASSISTANT PROFESSORS DATASETS CELL

In [0]:
n_iter = 1000
columns = ['GENDER', 'BIRTHYEAR', 'INSTITUTION', 'CITY']

def summary_stats(arr):
    mean = arr.mean()
    std = arr.std(ddof=1)
    ci = stats.norm.interval(0.95, loc=mean, scale=std / np.sqrt(n_iter))
    return mean, ci

def genreate_results(dataset):
    results = []

    for col in columns:
        y_true = dataset[col].values
        n = len(y_true)

        def evaluate_predictions(y_pred, distance=None):
            if distance is None:
                match = y_pred == y_true
            else:
                match = np.abs(y_pred - y_true) <= distance

            accuracy = np.mean(match)
            precision = precision_score(y_true, y_pred, average='weighted', zero_division=0)
            recall = recall_score(y_true, y_pred, average='weighted', zero_division=0)

            return accuracy, precision, recall

        def simulate(distance=None):
            accuracy_list = []
            precision_list = []
            recall_list = []

            for _ in range(n_iter):
                y_pred = np.random.choice(y_true, size=n, replace=True)
                accuracy, precision, recall = evaluate_predictions(y_pred, distance)
                accuracy_list.append(accuracy)
                precision_list.append(precision)
                recall_list.append(recall)

            mean_acc, ci_acc = summary_stats(np.array(accuracy_list))
            mean_prec, ci_prec = summary_stats(np.array(precision_list))
            mean_rec, ci_rec = summary_stats(np.array(recall_list))

            return {
                'Feature': col if distance is None else f'{col} (±{distance})',
                'Accuracy (95% CI)': f"{mean_acc * 100:.2f} ({ci_acc[0] * 100:.2f}, {ci_acc[1] * 100:.2f})",
                'Precision (95% CI)': f"{mean_prec * 100:.2f} ({ci_prec[0] * 100:.2f}, {ci_prec[1] * 100:.2f})",
                'Recall (95% CI)': f"{mean_rec * 100:.2f} ({ci_rec[0] * 100:.2f}, {ci_rec[1] * 100:.2f})",
            }

        results.append(simulate())

        if col == 'BIRTHYEAR':
            results.append(simulate(distance=1))
            results.append(simulate(distance=2))

    return pd.DataFrame(results)

In [0]:
display(genreate_results(DATASET_FULL_PROFESSORS))

In [0]:
display(genreate_results(DATASET_ASSISTANT_PROFESSORS))